# Hospital Transfer Network — Monte Carlo SI Model

In [41]:
import pandas as pd
import numpy as np
import networkx as nx
from collections import defaultdict


# ── CELL 1: Load data

In [42]:
pdf     = pd.read_csv("agg_hospital_transfers.csv")   # source, target, weight
comm_df = pd.read_csv("hospital_communities.csv")     # hospital, community_id, ...

BETA           = 1  # transmission rate per transfer event
N_RUNS         = 100  # Monte Carlo runs per seed node

print(f"Hospitals: {len(comm_df)}")
print(f"Transfer pairs (raw): {len(pdf)}")

Hospitals: 125
Transfer pairs (raw): 8691


# ── CELL 2: Filter to community hospitals only

In [43]:
# Only keep edges where both source AND target are in comm_df
community_nodes = set(comm_df["hospital"].tolist())

pdf_filtered = pdf[
    pdf["source"].isin(community_nodes) &
    pdf["target"].isin(community_nodes)
].copy()

print(f"Transfer pairs (filtered to community hospitals): {len(pdf_filtered)}")


Transfer pairs (filtered to community hospitals): 8691


# ── CELL 3: Build directed graph

In [44]:
G = nx.from_pandas_edgelist(
    pdf,
    source="source",
    target="target",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Graph: 125 nodes, 5265 edges


# ── CELL 4: Normalise edge weights to transmission probabilities

In [45]:
# For each sending node, divide each edge weight by total out-weight
# P(A→B) = weight(A→B) / sum of all weights leaving A
# Final transmission probability per step = β × P(A→B)

for node in G.nodes():
    total_out = sum(d.get("weight", 1) for _, _, d in G.out_edges(node, data=True))
    for _, neighbor, data in G.out_edges(node, data=True):
        data["trans_prob"] = BETA * (data["weight"] / total_out) if total_out > 0 else 0.0

print("\nSample edge transmission probabilities:")
for u, v, d in list(G.edges(data=True))[:5]:
    print(f"  {u} → {v}  weight={d['weight']}  trans_prob={d['trans_prob']:.4f}")



Sample edge transmission probabilities:
  RJE → RL4  weight=220  trans_prob=0.0897
  RJE → RBK  weight=37  trans_prob=0.0151
  RJE → RQ3  weight=60  trans_prob=0.0245
  RJE → RVY  weight=1  trans_prob=0.0004
  RJE → RBS  weight=16  trans_prob=0.0065


# ── CELL 5: SI simulation function

In [46]:
def run_si(G, seed_node):
    """
    Run a single SI simulation from a given seed node.
    Timesteps are abstract transmission cycles — not calendar time.

    Returns:
        steps_to_infection : dict {node: step first infected}  (seed = step 0)
        final_size         : total infected nodes at end
    """
    infected            = {seed_node}
    susceptible         = set(G.nodes()) - infected
    steps_to_infection  = {seed_node: 0}
    step                = 0

    while True:
        newly_infected = set()

        for node in infected:
            for _, neighbor, data in G.out_edges(node, data=True):
                if neighbor in susceptible:
                    if np.random.random() < data["trans_prob"]:
                        newly_infected.add(neighbor)

        if not newly_infected:
            break  # no new infections this cycle — simulation ends

        step += 1
        infected   |= newly_infected
        susceptible -= newly_infected
        for node in newly_infected:
            steps_to_infection[node] = step

    return steps_to_infection, len(infected)


# ── CELL 6: Monte Carlo — run for every seed node

In [47]:
nodes   = list(G.nodes())
n_nodes = len(nodes)

# infection_counts[node]  = times this node got infected across ALL runs
# seeding_sizes[node]     = epidemic sizes when THIS node was the seed
# infection_steps[node]   = steps to infection across all runs (excluding seed runs)

infection_counts = defaultdict(int)
seeding_sizes    = defaultdict(list)
infection_steps  = defaultdict(list)

total_runs = n_nodes * N_RUNS
run_count  = 0

print(f"\nRunning Monte Carlo SI model...")
print(f"  Seed nodes:  {n_nodes}")
print(f"  Runs/seed:   {N_RUNS}")
print(f"  Total runs:  {total_runs}")
print(f"  β:           {BETA}\n")

for seed in nodes:
    for _ in range(N_RUNS):
        steps_dict, final_size = run_si(G, seed)

        seeding_sizes[seed].append(final_size)

        for node, step in steps_dict.items():
            infection_counts[node] += 1
            if node != seed:
                infection_steps[node].append(step)

        run_count += 1
        if run_count % 1000 == 0:
            print(f"  {run_count}/{total_runs} runs complete...")

print(f"  {total_runs}/{total_runs} runs complete.\n")


Running Monte Carlo SI model...
  Seed nodes:  125
  Runs/seed:   100
  Total runs:  12500
  β:           1

  1000/12500 runs complete...
  2000/12500 runs complete...
  3000/12500 runs complete...
  4000/12500 runs complete...
  5000/12500 runs complete...
  6000/12500 runs complete...
  7000/12500 runs complete...
  8000/12500 runs complete...
  9000/12500 runs complete...
  10000/12500 runs complete...
  11000/12500 runs complete...
  12000/12500 runs complete...
  12500/12500 runs complete.



# ── CELL 7: Compute output metrics per node

In [48]:
results = []

for node in nodes:
    # Infection frequency: proportion of ALL runs where this node got infected
    inf_freq = infection_counts[node] / total_runs

    # Seeding impact: mean epidemic size when THIS node is the index case
    mean_epi_size     = np.mean(seeding_sizes[node])
    mean_epi_size_pct = mean_epi_size / n_nodes * 100

    # Mean step to infection across all runs where it was infected (not as seed)
    steps     = infection_steps[node]
    mean_step = np.mean(steps) if steps else np.nan

    results.append({
        "hospital":               node,
        "infection_frequency":    round(inf_freq,         4),
        "mean_epidemic_size":     round(mean_epi_size,    1),
        "mean_epidemic_size_pct": round(mean_epi_size_pct,1),
        "mean_step_to_infection": round(mean_step,        2) if not np.isnan(mean_step) else np.nan,
    })

si_df = pd.DataFrame(results).sort_values("infection_frequency", ascending=False)

# ── CELL 8: Merge with community info

In [49]:
si_df = si_df.merge(
    comm_df[["hospital", "provider", "community_id",
             "weighted_in_degree", "cross_community_bridges"]],
    on="hospital",
    how="left"
)

print("si_df hospital sample:  ", si_df["hospital"].head(3).tolist())
print("comm_df hospital sample:", comm_df["hospital"].head(3).tolist())
print("si_df dtypes:\n",   si_df.dtypes)
print("comm_df dtypes:\n", comm_df.dtypes)
print("Overlap:", len(set(si_df["hospital"]) & set(comm_df["hospital"])))

si_df hospital sample:   ['RHQ', 'RJ1', 'RJZ']
comm_df hospital sample: ['R0A', 'RWJ', 'REM']
si_df dtypes:
 hospital                    object
infection_frequency        float64
mean_epidemic_size         float64
mean_epidemic_size_pct     float64
mean_step_to_infection     float64
provider                    object
community_id                 int64
weighted_in_degree           int64
cross_community_bridges      int64
dtype: object
comm_df dtypes:
 hospital                   object
provider                   object
community_id                int64
weighted_in_degree          int64
weighted_out_degree         int64
total_transfers             int64
cross_community_bridges     int64
dtype: object
Overlap: 125


# ── CELL 9: Print results

In [50]:
print("=" * 75)
print(f"Monte Carlo SI Results  |  β={BETA}  |  {N_RUNS} runs/seed  |  {n_nodes} nodes")
print("=" * 75)

print("\n--- Top 20: Infection Frequency ---")
print("Hospitals most often infected regardless of where outbreak starts\n")
print(si_df.head(20)[[
    "hospital", "provider", "community_id",
    "infection_frequency", "mean_step_to_infection",
    "weighted_in_degree", "cross_community_bridges"
]].to_string(index=False))

print("\n--- Top 20: Seeding Impact ---")
print("If THIS hospital is the index case, how many eventually get infected\n")
print(
    si_df.sort_values("mean_epidemic_size", ascending=False).head(20)[[
        "hospital", "provider", "community_id",
        "mean_epidemic_size", "mean_epidemic_size_pct", "infection_frequency"
    ]].to_string(index=False)
)

print("\n--- Community Vulnerability ---")
print(
    si_df.groupby("community_id").agg(
        hospitals               =("hospital",               "count"),
        avg_infection_frequency =("infection_frequency",    "mean"),
        avg_epidemic_size       =("mean_epidemic_size",     "mean"),
        avg_step_to_infection   =("mean_step_to_infection", "mean"),
    ).round(3).to_string()
)

Monte Carlo SI Results  |  β=1  |  100 runs/seed  |  125 nodes

--- Top 20: Infection Frequency ---
Hospitals most often infected regardless of where outbreak starts

hospital                                                 provider  community_id  infection_frequency  mean_step_to_infection  weighted_in_degree  cross_community_bridges
     RHQ        SHEFFIELD TEACHING HOSPITALS NHS FOUNDATION TRUST            10               0.4729                    4.46               87382                      159
     RJ1                GUY'S AND ST THOMAS' NHS FOUNDATION TRUST             2               0.4492                    4.40               52240                      159
     RJZ             KING'S COLLEGE HOSPITAL NHS FOUNDATION TRUST             2               0.4361                    5.27               25869                      145
     RFF                   BARNSLEY HOSPITAL NHS FOUNDATION TRUST            10               0.4340                    6.37                3554         

# ── CELL 10: Save

In [51]:
si_df.to_csv("hospital_si_results.csv", index=False)
print("\nSaved to: hospital_si_results.csv")
print("Columns:", si_df.columns.tolist())


Saved to: hospital_si_results.csv
Columns: ['hospital', 'infection_frequency', 'mean_epidemic_size', 'mean_epidemic_size_pct', 'mean_step_to_infection', 'provider', 'community_id', 'weighted_in_degree', 'cross_community_bridges']
